## Overfitting and Underfitting Analysis

### 1. Motivation 

The primary goal of this project is to examine whether incorporating neighbourhood-based statistical features can improve house price prediction performance.

However, evaluating model quality using **Test Mean Squared Error (MSE) alone is insufficient**.  
A lower Test MSE may arise from two different causes:

1. genuine improvement in model generalisation, or  
2. overfitting, where the model memorises training data but fails to generalise.

To distinguish between these possibilities, we compare **Train MSE and Test MSE** for each model.  
The relative difference between these two metrics provides an indication of the model's generalisation behaviour.


### 2. Diagnostic Method 

We define the **Test-to-Train MSE Ratio** as a simple diagnostic indicator:

Ratio = Test MSE / Train MSE

Diagnosis is based on **relative comparison rather than absolute error thresholds**,  
since the absolute magnitude of MSE depends on the scale of the dataset.

| Condition | Interpretation |
|---|---|
| Ratio > 1.5 | Possible overfitting — test error substantially larger than training error |
| Ratio < 1.1 | Small generalisation gap — if both errors remain high, this may indicate underfitting |
| Otherwise | Reasonable fit — no large train–test gap observed |

The value **1.5** is used as a heuristic diagnostic threshold.  
If the test error exceeds the training error by more than approximately 50%, the model may exhibit signs of overfitting.

In [ ]:
# ============================================================
# Overfitting / Underfitting Check
#
# Motivation:
#   Reporting Test MSE alone is insufficient to evaluate model quality.
#   A lower Test MSE could result from genuine generalisation improvement,
#   or simply from the model overfitting the training data.
#   By comparing Train MSE and Test MSE, we can assess whether the
#   improvement in Model 2 is more likely to reflect better generalisation
#   rather than a larger train–test gap.
#
# Why Train vs Test (not Train vs Val):
#   In the final training stage, the training and validation sets are
#   merged together to maximise the amount of data available for training.
#   This means no independent validation set remains.
#   The held-out test set therefore serves as the reference for
#   evaluating generalisation performance.
# ============================================================
import matplotlib.pyplot as plt
import numpy as np

def diagnose_final(model_name, train_mse, test_mse, threshold_ratio=1.5):  # heuristic threshold
    """
    Print an overfitting/underfitting diagnosis based on Test/Train MSE ratio.

    Diagnosis rules:
        ratio > 1.5  → Possible overfitting
        ratio < 1.1  → Low generalisation gap; possible underfitting if both errors remain high
        Otherwise    → Reasonable fit
    """
    # Compute ratio; set to infinity if train_mse is zero to avoid division by zero
    ratio = test_mse / train_mse if train_mse > 0 else float('inf')

    print(f"\n{'='*55}")
    print(f"  {model_name}")
    print(f"{'='*55}")
    print(f"  Train MSE       : {train_mse:,.4f}")
    print(f"  Test  MSE       : {test_mse:,.4f}")
    print(f"  Test/Train Ratio: {ratio:.3f}")

    # Diagnosis uses relative comparison rather than absolute thresholds,
    # as the appropriate MSE scale is dataset-dependent.
    if ratio > threshold_ratio:
        # Test MSE notably larger than Train MSE → possible overfitting
        print("  ⚠ Diagnosis : POSSIBLE OVERFITTING — test MSE is notably larger than train MSE")
    elif ratio < 1.1:
        # Train and test MSE are similar but no absolute threshold is used,
        # as MSE scale is dataset-dependent.
        print("  ~ Diagnosis : LOW GENERALISATION GAP — train and test errors are similar")
    else:
        # Train and test MSE are reasonably close → no large generalisation gap
        print("  ✓ Diagnosis : REASONABLE FIT — no large train–test gap is observed")
    print(f"{'='*55}")

In [ ]:
# ============================================================
# MODEL 1 — Final Baseline
#
# baseline_final_model was trained on the merged train+val set
# using only the original features, with no neighbourhood features.
# baseline_train_mse and baseline_test_mse_final were already computed
# immediately after training, so we reuse them directly here.
# ============================================================

m1_train_mse = baseline_train_mse        # already computed above
m1_test_mse  = baseline_test_mse_final   # already computed above

diagnose_final("Model 1 — Final Baseline", m1_train_mse, m1_test_mse)



# ============================================================
# MODEL 2 — Final Enhanced
#
# model_enh_final was trained on the merged train+val set
# with neighbourhood-based price statistics added as extra inputs.
# enhanced_test_mse_final was already computed immediately after training.
# Train MSE was not computed in the original code, so we compute it
# once here using the already-fitted model. No re-training is required.
# ============================================================

m2_test_mse  = enhanced_test_mse_final   # already computed above

# Train MSE for Model 2 was not computed in the original code.
# Computed once here using the already-fitted model_enh_final.
m2_train_mse = mean_squared_error(        # not in original code, compute once here
    y_train_final,
    model_enh_final.predict(X_train_enh_final)
)

diagnose_final("Model 2 — Final Enhanced", m2_train_mse, m2_test_mse)

### 3. Results 

Both models exhibit **relatively small train–test gaps**, suggesting that neither model shows strong evidence of overfitting or severe underfitting.

Although Model 2 shows a slightly larger Test/Train ratio, the value remains well below the overfitting threshold, indicating that the improvement is unlikely to be caused by excessive memorisation of the training data.

Overall, the results suggest that neighbourhood-based features contribute to improved generalisation performance without introducing clear signs of overfitting.

In [ ]:
# ============================================================
# VISUALISATION 1 — Bar chart: Train vs Test MSE
#
# Purpose:
#   Visually compare the gap between Train MSE and Test MSE for both
#   models. A larger gap may indicate possible overfitting.
# ============================================================

labels     = ['Model 1\n(Baseline)', 'Model 2\n(Enhanced)']
train_mses = [m1_train_mse, m2_train_mse]
test_mses  = [m1_test_mse,  m2_test_mse]

# X-axis positions for each group of bars
x     = np.arange(len(labels))
width = 0.35  # Width of each individual bar

fig, ax = plt.subplots(figsize=(8, 5))

# Plot Train MSE bars on the left of each group
bars1 = ax.bar(x - width/2, train_mses, width, label='Train MSE', color='steelblue',  alpha=0.85)

# Plot Test MSE bars on the right of each group
bars2 = ax.bar(x + width/2, test_mses,  width, label='Test MSE',  color='darkorange', alpha=0.85)

ax.set_ylabel('MSE')
ax.set_title('Train vs Test MSE — Model 1 and Model 2')
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.legend()

# Annotate each bar with its exact MSE value
# We use ax.text() instead of bar_label() for compatibility
# with older versions of matplotlib
for bar, val in zip(bars1, train_mses):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
            f'{val:.4f}', ha='center', va='bottom', fontsize=9)
for bar, val in zip(bars2, test_mses):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
            f'{val:.4f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()


# ============================================================
# VISUALISATION 2 — Learning Curves (Train MSE vs Epoch)
# 可视化2 — 学习曲线（训练集MSE随训练轮数变化）
#
# Purpose / 目的:
#   The ratio test above evaluates only the final model state.
#   To provide additional insight, learning curves are plotted
#   to illustrate how Train MSE evolved during the actual training
#   of the final models.
#
#   上述比值检验仅反映模型最终状态。
#   为进一步观察训练过程，这里绘制学习曲线，
#   展示最终模型在真实训练过程中训练误差的变化趋势。
#
# Why loss_curve_ / 为什么使用 loss_curve_:
#   loss_curve_ records the training loss used by the optimiser
#   (squared error loss), which equals MSE / 2. Multiplying by 2
#   converts it to true MSE for consistent comparison with test MSE.
#
#   loss_curve_ 记录的是优化器使用的训练损失（squared error loss），
#   等于 MSE / 2。乘以 2 将其转换为真实 MSE，
#   与测试集 MSE 保持单位一致。
#
# Design choice / 设计说明:
#   Only Train MSE is plotted per epoch. The final Test MSE is
#   shown as a single horizontal reference line to avoid using
#   the test set as an epoch-level model selection reference.
#
#   每个 epoch 仅绘制训练集MSE曲线。
#   测试集MSE仅以水平参考线展示，
#   以避免在训练过程中将测试集用于模型选择。
# ============================================================

# Retrieve actual training history and convert loss to MSE
# loss_curve_ = squared_error / 2, so multiply by 2 to obtain true MSE
# 取出真实训练历史并将损失转换为 MSE
# loss_curve_ = squared_error / 2，乘以 2 得到真实 MSE
train_loss_curve_m1 = np.array(baseline_final_model.loss_curve_) * 2
train_loss_curve_m2 = np.array(model_enh_final.loss_curve_) * 2

# Print actual number of training epochs for each model
# Fewer epochs than max_iter indicates optimiser convergence before max_iter
# 打印每个模型的实际训练轮数
# 若少于 max_iter，说明优化器在达到最大轮数之前已收敛
print(f"Model 1 actual training epochs: {len(train_loss_curve_m1)}")
print(f"Model 2 actual training epochs: {len(train_loss_curve_m2)}")
print("Note: fewer epochs indicate optimiser convergence before max_iter.")
print("注：轮数少于 max_iter 说明优化器提前收敛。")

# Plot learning curves for both models side by side
# 并排绘制两个模型的学习曲线
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
# Unify y-axis range across both subplots for fair visual comparison
# 统一两个子图的 y 轴范围，确保视觉比较的公平性
global_min = min(train_loss_curve_m1.min(), train_loss_curve_m2.min(), m1_test_mse, m2_test_mse)
global_max = max(train_loss_curve_m1.max(), train_loss_curve_m2.max())

for idx, (ax, lc_tr, final_test_mse, final_train_mse, title) in enumerate(zip(
    axes,
    [train_loss_curve_m1, train_loss_curve_m2],
    [m1_test_mse,  m2_test_mse],
    [m1_train_mse, m2_train_mse],
    ['Model 1 — Final Baseline', 'Model 2 — Final Enhanced']
)):
    epochs = range(1, len(lc_tr) + 1)

    # Solid blue line: Train MSE recorded at each epoch during actual training
    # 蓝色实线：真实训练过程中每个 epoch 记录的训练集 MSE
    ax.plot(epochs, lc_tr, label='Train MSE', color='steelblue')

    # Dotted blue line: Final Train MSE shown as reference
    # 蓝色点线：最终训练集 MSE，作为参考线
    ax.axhline(
        y=final_train_mse,
        color='steelblue',
        linestyle=':',
        linewidth=1.2,
        label=f'Final Train MSE = {final_train_mse:.4f}'
    )

    # Horizontal dashed orange line: final Test MSE shown as reference only
    # The test set is evaluated once at the end, not per epoch
    # 橙色水平虚线：最终测试集 MSE，仅作参考
    # 测试集只在训练结束后评估一次，不参与逐 epoch 计算
    ax.axhline(
        y=final_test_mse,
        color='darkorange',
        linestyle='--',
        linewidth=1.2,
        label=f'Final Test MSE = {final_test_mse:.4f}'
    )

    # Interpretation / 曲线解读:
    #   Train curve converging toward the test reference → good generalisation
    #   Train curve far below the test reference → may indicate potential overfitting
    #   Both train curve and test line remaining high → possible underfitting
    #   if both errors remain relatively high compared with earlier model configurations
    #
    #   训练曲线接近测试参考线 → 泛化良好
    #   训练曲线明显低于测试参考线 → 可能存在过拟合迹象
    #   两者均处于高位（相比早期模型配置）→ 可能欠拟合
    ax.set_ylim(global_min * 0.9, global_max * 1.05)
    ax.set_title(title)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('MSE')
    ax.legend(fontsize=8)
    ax.grid(True, linestyle=':', alpha=0.6)

plt.suptitle(
    'Learning Curves — Overfitting / Underfitting Check',
    fontsize=13
)
plt.tight_layout()
plt.show()

### Figure Caption 

**Figure 1 — Train vs Test MSE Comparison**

Bar chart comparing the training and test mean squared errors of the two final models.  
The gap between Train MSE and Test MSE provides a visual indication of potential overfitting: a small gap suggests better generalisation beyond the training data.

**Figure 2 — Approximate Learning Curves**

Approximate learning curves showing how the training MSE evolves during optimisation.  
The solid line represents the training error trajectory, while the dashed horizontal line indicates the final test error as a reference.

These curves provide a qualitative view of the optimisation behaviour rather than exact epoch-level training logs.



### 4. Learning Curve Analysis 

In addition to the final MSE comparison, **approximate learning curves** are plotted to visualise how the training error evolves during optimisation.

The curves are generated by **re-simulating training from scratch** using `warm_start=True` with `max_iter=1` in each call to `fit()`.

Although the architecture, hyperparameters, and random seed are identical to those of the final models, this procedure **does not reproduce the exact training history**.

In sklearn's `MLPRegressor`, each call to `fit(max_iter=1)` **does not strictly correspond to one epoch**, due to optimiser internal states and stochastic data shuffling.

Therefore, the curves should be interpreted as **approximate optimisation trajectories**.

Only the **training error** is recorded per iteration, while the final **test error** is shown as a horizontal reference line.  
This avoids repeatedly evaluating the test set during training and helps prevent methodological bias.